# Avenue Pooling Diagnosis — where does the 64% ceiling come from?

Two completely different pose stacks (AlphaPose vs YOLO26+ByteTrack+ViTPose++) both land on
**~64% Micro AUC** on Avenue. ViTPose++ keypoints are measurably better on small/occluded people,
and swapping stacks changed nothing — so the constraint is likely **downstream of extraction**, in
how per-person segment scores become a frame-level anomaly score.

This notebook tests that hypothesis with **existing artifacts only**: no re-extraction, no
retraining, no tracker change. It reuses the locked 64.14% checkpoint
(`logs/Avenue/64_2/checkpoint_64_2.pth.tar`, epoch 3) and the exact control scoring path
(`score_dataset`), then replaces the global min-pool over people with alternative aggregations
(size-gated min, size-weighted min, k-th smallest, confidence-gated min, confidence-weighted min)
and with *simulated stricter detection* (deleting garbage person-frames from the JSONs and
re-scoring).

Design doc: `docs/superpowers/specs/2026-08-26-avenue-pooling-diagnosis-design.md`

## Decision rule

1. **Baseline sanity:** the rebuilt per-clip pipeline must reproduce ~64.1421% Micro AUC.
2. **Part C (simulated stricter detection) gains ≥ 1–2 AUC pts** → extraction-side confirmation;
   the thresholds are already known from the simulation, so a real re-extraction is next.
3. **Part B (pooling variant) gains ≥ 1–2 AUC pts** (best Part C filter held fixed) → scoring-side
   confirmation; the next spec designs the full scoring experiment.
4. **All flat** → scoring aggregation and detection filtering are ruled out cheaply; the
   constraint is identity continuity (BoT-SORT/ReID + track stitching) or model capacity.
5. The **Part D reason histogram** explains *why*: `GARBAGE_DET`-driven gains confirm Part C,
   `CROWD_JITTER`-driven gains confirm size weighting, `OCCLUSION_GAP`/`ID_SWITCH` dominance
   redirects to the tracking path.

## Honest caveats

- Variant AUCs use a new scoring convention: they establish a **ceiling**, not a headline number
  comparable to the reported 64.2%.
- Keypoint-bbox height is a **size proxy**, not ground truth.
- The baseline path is kept byte-identical to the control run; only the pooling step changes.


## Step 0 — Setup

Mount Drive (poses and checkpoint live there), clone/update the STG-NF repo, install pure-Python
deps. No CUDA compilation needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Hadi6618/STG-NF.git /content/STG-NF 2>/dev/null || (cd /content/STG-NF && git pull)
!pip -q install scipy scikit-learn pandas matplotlib

import os, sys, json, shutil
from pathlib import Path
import numpy as np

REPO_DIR = Path("/content/STG-NF")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo ready:", REPO_DIR)

### Configuration

All paths and diagnostic thresholds are here — edit once. Thresholds are recorded in the run
manifest at the end so every number in the report is reproducible.

In [ ]:
# ---- Paths (control run: YOLO26 + ByteTrack + ViTPose++ base, Micro AUC 64.1421%) ----
DRIVE_ROOT   = Path("/content/drive/MyDrive/STG-NF/Avenue_dataset_vitpose")
CHECKPOINT   = DRIVE_ROOT / "logs/Avenue/64_2/checkpoint_64_2.pth.tar"
POSE_TRAIN   = DRIVE_ROOT / "pose/train"
POSE_TEST    = DRIVE_ROOT / "pose/test"
VIDEO_TRAIN  = Path("/content/avenue_dataset/Avenue Dataset/training_videos")
VIDEO_TEST   = Path("/content/avenue_dataset/Avenue Dataset/testing_videos")
GT_SRC       = Path("/content/drive/MyDrive/ground_truth_avenue")   # {1..21}.npy, 1 = normal / 0 = abnormal
OUTPUT_ROOT  = Path("/content/pooling_diagnosis")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional second stack for the Part D two-stack attribution; set to None to skip.
ALPHAPOSE_POSE_ROOT = Path("/content/drive/MyDrive/STG-NF/Avenue_dataset/pose")

# ---- Control-run hyper-parameters (must match the checkpoint) ----
SEG_LEN       = 24
SMOOTH_SIGMA  = 7          # score_dataset applies six cumulative passes, sigma = 1..6
ATTENTION     = "triplet"
N_HEADS, N_MECATT, N_MECATT_INSIDE = 1, 1, 1
ATTN_LR_MULT  = 1.0
ATTN_PROJ_TYPE = "bottleneck"
ATTN_BOTTLENECK = 64
BATCH_SIZE, NUM_WORKERS = 256, 2

# ---- Diagnostic thresholds (stated explicitly for reproducibility) ----
KP_CONF_SIZE   = 0.25      # keypoint confidence floor for keypoint-bbox size estimation
SIZE_GATES     = [80, 120, 160]   # Part B size gate, px
SIZE_REF       = 160.0     # reference size for size-weighted pooling (foreground threshold)
SIZE_ALPHAS    = [0.5, 1.0]        # Part B size-weight exponent
KTH_LIST       = [2, 3, 5]         # Part B robust k-th smallest (diagnostic probe only)
CONF_TAUS      = [0.3, 0.5, 0.7]   # Part B confidence gate

PART_C_DET_TAUS   = [0.4, 0.5, 0.7]   # Part C detection-confidence floor
PART_C_KP_FLOORS  = [6, 10]           # Part C confident-keypoint floor
PART_C_SIZE_FLOORS = [60, 80]         # Part C minimum person size, px
PART_C_TRACK_LENS = [10, 24]          # Part C minimum track length, frames
KP_CONF_FILTER    = 0.3               # keypoint confidence floor for the Part C kp-quality filter
PART_C_COMBINED   = dict(det_tau=0.5, kp_floor=6, size_floor=60, min_track=10)

TOP_FP_FRAC  = 0.01        # top-1% of frames flagged as false positives (min MIN_FP_N frames)
MIN_FP_N     = 100
CROWD_THETA  = 80          # size below which + high jitter => CROWD_JITTER reason code
JITTER_HIGH  = 8.0         # mean frame-to-frame keypoint displacement (px) above which = jittery
GARBAGE_CONF = 0.3         # det confidence below which => GARBAGE_DET
GARBAGE_KP   = 6           # confident keypoints below which => GARBAGE_DET
MISSED_GAP   = 8           # interpolation window for the gap probe (frames)
RUN_GAP_PROBE = True       # set False to skip the gap-interpolation model pass

print("checkpoint:", CHECKPOINT)
print("output root:", OUTPUT_ROOT)

In [ ]:
# ---- Ground truth: copy Drive {1..21}.npy into the repo layout score_dataset expects ----
def prepare_gt():
    gt_dst = REPO_DIR / "data/Avenue/gt/test_frame_mask"
    gt_dst.mkdir(parents=True, exist_ok=True)
    for n in range(1, 22):
        src = GT_SRC / f"{n}.npy"
        dst = gt_dst / f"01_{n:04d}.npy"
        if not dst.exists() and src.exists():
            shutil.copy2(src, dst)
    files = sorted(gt_dst.glob("*.npy"))
    print("GT files ready:", len(files))
    return gt_dst

GT_DIR = prepare_gt()

### Scoring library (control path, verbatim)

These helpers mirror `stgnf_export_scores.py` exactly: same args, same `get_dataset_and_loader`,
same `Trainer.test()`, same `score_dataset`. The baseline produced by this path **must** reproduce
64.1421% — that is the pipeline-integrity check everything else builds on.

In [ ]:
import torch
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import roc_auc_score
import pandas as pd

from args import init_parser, init_sub_args
from dataset import get_dataset_and_loader, PoseSegDataset
from models.STG_NF.model_pose import STG_NF
from models.training import Trainer
from utils.data_utils import trans_list
from utils.optim_init import init_optimizer, init_scheduler
from utils.train_utils import init_model_params
from utils.scoring_utils import score_dataset, gt_root_for_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", device)


def build_args(pose_test_dir=None):
    argv = [
        "--dataset", "Avenue",
        "--pose_path_train", str(POSE_TRAIN),
        "--pose_path_test", str(pose_test_dir or POSE_TEST),
        "--vid_path_train", str(VIDEO_TRAIN),
        "--vid_path_test", str(VIDEO_TEST),
        "--checkpoint", str(CHECKPOINT),
        "--seg_len", str(SEG_LEN),
        "--batch_size", str(BATCH_SIZE),
        "--num_workers", str(NUM_WORKERS),
        "--attention", ATTENTION,
        "--n_heads", str(N_HEADS),
        "--n_mecatt", str(N_MECATT),
        "--n_mecatt_inside", str(N_MECATT_INSIDE),
        "--attention_lr_mult", str(ATTN_LR_MULT),
        "--attention_proj_type", ATTN_PROJ_TYPE,
        "--attention_bottleneck_dim", str(ATTN_BOTTLENECK),
    ]
    args = init_parser().parse_args(argv)
    args, _ = init_sub_args(args)
    return args


def run_control(pose_test_dir=None, load_ckpt=True):
    """Rebuild the control pipeline and return (per-segment scores, dataset, args)."""
    args = build_args(pose_test_dir)
    dataset, loader = get_dataset_and_loader(args, trans_list=trans_list, only_test=True)
    model_args = init_model_params(args, dataset)
    model = STG_NF(**model_args)
    trainer = Trainer(
        args, model, loader["train"], loader["test"],
        optimizer_f=init_optimizer(args.model_optimizer, lr=args.model_lr),
        scheduler_f=init_scheduler(args.model_sched, lr=args.model_lr, epochs=args.epochs),
    )
    if load_ckpt:
        trainer.load_checkpoint(str(CHECKPOINT))
    scores = trainer.test()
    return scores, dataset, args


def per_segment_size_conf(dataset):
    """Per-segment p90 keypoint-bbox height + mean detection confidence, aligned with metadata.

    segs_data_np is (N, 3, T, V) raw pixel coords (ch0=x, ch1=y, ch2=conf), so the y-extent
    over confident keypoints is the person size proxy. p90 over the segment's frames keeps
    mid-stride poses from distorting the estimate.
    """
    data = dataset["test"].segs_data_np        # (N, 3, T, V)
    score = dataset["test"].segs_score_np      # (N, T) per-frame det confidence
    conf = data[:, 2]
    y = data[:, 1]
    mask = conf >= KP_CONF_SIZE
    ys = np.where(mask, y, np.nan)
    frame_h = np.nanmax(ys, axis=-1) - np.nanmin(ys, axis=-1)   # (N, T)
    seg_h = np.nanpercentile(frame_h, 90, axis=1)               # (N,)
    seg_c = score.mean(axis=1)
    return seg_h, seg_c


def build_split_dataset(pose_dir, stride, seg_len=SEG_LEN):
    """CPU-only PoseSegDataset for a split (no model pass). Used for segment counting in Part A/C."""
    ds_args = {
        "headless": False, "scale": 0, "scale_proportional": 1,
        "seg_len": seg_len, "return_indices": True, "return_metadata": True,
        "dataset": "Avenue", "train_seg_conf_th": 0.0, "specific_clip": None,
        "seg_stride": stride, "vid_path": None, "trans_list": None,
    }
    return PoseSegDataset(str(pose_dir), path_to_vid_dir=None,
                          normalize_pose_segs=True, evaluate=(stride == 1), **ds_args)

print("library ready")
print("sanity: segs_data_np channel convention is [x, y, conf] (raw pixels)")

---
# Part A — JSON-only statistics (no GPU)

**Question:** does the flow train predominantly on *crowd-sized* people while the test anomalies
live at *foreground sizes*? And how much of the test segment mass is low-confidence?

Computes per-segment p90 keypoint-bbox height and mean detection confidence for **train vs test**
and reports the crowd/foreground split using the same θ thresholds as Part B. No model pass.

In [ ]:
def part_a_stats():
    rows = []
    for split, pose_dir, stride in [("train", POSE_TRAIN, 6), ("test", POSE_TEST, 1)]:
        ds = build_split_dataset(pose_dir, stride)
        h, c = per_segment_size_conf({"test": ds})
        for i in range(len(h)):
            rows.append({"split": split, "p90_height": float(h[i]), "mean_conf": float(c[i])})
    df = pd.DataFrame(rows)
    return df

a_df = part_a_stats()
a_df.to_csv(OUTPUT_ROOT / "part_a_segment_stats.csv", index=False)

for split in ["train", "test"]:
    sub = a_df[a_df["split"] == split]
    print("=" * 66)
    print(f"{split.upper()}  segments={len(sub)}")
    print(f"  p90 height:  median={sub['p90_height'].median():7.1f}  p25={sub['p90_height'].quantile(0.25):7.1f}  p75={sub['p90_height'].quantile(0.75):7.1f}")
    print(f"  crowd (<{SIZE_GATES[0]}px):  {(sub['p90_height'] < SIZE_GATES[0]).mean()*100:5.1f}%")
    print(f"  mid        :  {((sub['p90_height'] >= SIZE_GATES[0]) & (sub['p90_height'] < SIZE_GATES[-1])).mean()*100:5.1f}%")
    print(f"  fg (>={SIZE_GATES[-1]}px):  {(sub['p90_height'] >= SIZE_GATES[-1]).mean()*100:5.1f}%")
    print(f"  mean conf:   median={sub['mean_conf'].median():.3f}  <0.5={(sub['mean_conf'] < 0.5).mean()*100:5.1f}%")
print("=" * 66)
print("saved:", OUTPUT_ROOT / "part_a_segment_stats.csv")

### Reading Part A

- If **train is dominated by crowd-sized segments** while test anomalies are large/close, the
  size-prior case is half-proven: the flow's learned "normal" distribution is shaped by small,
  jittery people, and the same kinematics at foreground scale read as unusual.
- If a large fraction of **test** segments is small (< 80 px) or low-confidence (< 0.5), that is
  the noise mass the pooling variants (Part B) and the detection filters (Part C) must remove.

---
# Part B — One model pass + alternative pooling

**One** model pass with the locked checkpoint (same code path as the control), then rebuild
per-person per-frame arrays exactly as `get_clip_score` does and replace the global `amin` with
each variant. Everything else (inf handling, six cumulative Gaussian passes, AUC computation)
stays byte-identical to `score_dataset`.

**Semantic note (VAD labels):** a frame is abnormal if *any* person is anomalous, so min-pooling
is the semantically correct aggregation — the lying person among five walkers must drive the
frame. The variants therefore keep the min (filter-then-min); `k-th smallest` is kept as a
diagnostic probe only (it violates VAD semantics) to quantify how much a single noise person
dominates the min.

In [ ]:
scores, dataset, args = run_control()

# --- Pipeline-integrity check: control micro AUC must reproduce 64.1421% ---
auc_control, _ = score_dataset(scores, dataset["test"].metadata, args=args)
print(f"Control Micro AUC (smoothed): {auc_control*100:.4f}%")
assert abs(auc_control - 0.641421) < 0.002, "control baseline drifted!"
print("OK: baseline reproduces the control run")

In [ ]:
def build_clip_views(scores, dataset, args):
    """Per clip: gt + per-person per-frame score / size / conf arrays (inf where no segment)."""
    meta = dataset["test"].metadata
    meta_np = np.array(meta)
    seg_h, seg_c = per_segment_size_conf(dataset)
    gt_root = gt_root_for_dataset(args.dataset)
    clip_list = sorted(fn for fn in os.listdir(gt_root) if fn.endswith(".npy"))
    views = {}
    for clip in clip_list:
        scene_id, clip_id = [int(i) for i in clip.replace("label", "001").split('.')[0].split('_')]
        inds = np.where((meta_np[:, 1] == clip_id) & (meta_np[:, 0] == scene_id))[0]
        gt = np.load(os.path.join(gt_root, clip)).astype(np.float64)
        gt = 1.0 - gt   # same convention as get_clip_score
        T = gt.shape[0]
        persons = sorted(set(int(meta_np[i, 2]) for i in inds))
        p_score = {p: np.full(T, np.inf) for p in persons}
        p_size  = {p: np.full(T, np.inf) for p in persons}
        p_conf  = {p: np.full(T, np.inf) for p in persons}
        for i in inds:
            p = int(meta_np[i, 2])
            frame = int(meta[i][3]) + args.seg_len // 2
            if 0 <= frame < T:
                p_score[p][frame] = scores[i]
                p_size[p][frame] = seg_h[i]
                p_conf[p][frame] = seg_c[i]
        views[clip] = {"gt": gt, "persons": persons,
                       "score": p_score, "size": p_size, "conf": p_conf}
    return views

views = build_clip_views(scores, dataset, args)
print("clips:", len(views))
tot_ppl = sum(len(v["persons"]) for v in views.values())
print("total distinct persons across test clips:", tot_ppl)

In [ ]:
def pool_variant(views, variant, **kw):
    """Replace the global min over people with a variant pooling. Output stays in the control
    convention (lower = more anomalous), so downstream smoothing + AUC are unchanged."""
    pooled = {}
    for clip, v in views.items():
        persons = v["persons"]
        score = np.stack([v["score"][p] for p in persons])   # (P, T)
        size  = np.stack([v["size"][p] for p in persons])    # (P, T)
        conf  = np.stack([v["conf"][p] for p in persons])    # (P, T)
        if variant == "baseline":
            out = score.min(axis=0)
        elif variant == "size_gate":
            keep = size >= kw["theta"]
            out = np.where(keep, score, np.inf).min(axis=0)
        elif variant == "size_weighted":
            alpha = kw["alpha"]
            w = np.clip(size / SIZE_REF, 0.0, None) ** alpha   # small persons pulled toward normal
            out = (score * w).min(axis=0)
        elif variant == "kth":
            k = min(int(kw["k"]), score.shape[0])           # guard: clips with < k people
            out = np.sort(score, axis=0)[k - 1]                # k-th most anomalous (probe only)
        elif variant == "conf_gate":
            keep = conf >= kw["tau"]
            out = np.where(keep, score, np.inf).min(axis=0)
        elif variant == "conf_weighted":
            out = (score * conf).min(axis=0)                   # low conf pulled toward normal
        else:
            raise ValueError(variant)
        pooled[clip] = out
    return pooled


def eval_variants(pooled, views, smooth=True):
    """Mirror get_dataset_scores + smooth_scores + score_auc exactly (control convention)."""
    gt_arr, sc_arr = [], []
    for clip, out in pooled.items():
        gt_arr.append(views[clip]["gt"])
        sc_arr.append(out.copy())
    scores_np = np.concatenate(sc_arr, axis=0)
    finite = scores_np[np.isfinite(scores_np)]
    if finite.size:
        scores_np[scores_np == np.inf] = finite.max()
        scores_np[scores_np == -np.inf] = finite.min()
    idx = 0
    for s in range(len(sc_arr)):
        for t in range(sc_arr[s].shape[0]):
            sc_arr[s][t] = scores_np[idx]
            idx += 1
    if smooth:
        for s in range(len(sc_arr)):
            for sig in range(1, SMOOTH_SIGMA):
                sc_arr[s] = gaussian_filter1d(sc_arr[s], sigma=sig)
    gt_np = np.concatenate(gt_arr)
    sc_np = np.concatenate(sc_arr)
    micro = roc_auc_score(gt_np, sc_np)
    # macro: per-video AUC, single-class clips skipped
    aucs = []
    for g, s in zip(gt_arr, sc_arr):
        if len(np.unique(g)) < 2:
            continue
        aucs.append(roc_auc_score(g, s))
    macro = float(np.mean(aucs)) if aucs else float("nan")
    return micro, macro, sc_arr, gt_arr


# Baseline sanity via the variant path: must equal the control micro AUC.
micro_base, macro_base, sc_arr, gt_arr = eval_variants(pool_variant(views, "baseline"), views)
print(f"Variant-path baseline: micro={micro_base*100:.4f}%  macro={macro_base*100:.4f}%")
assert abs(micro_base - auc_control) < 1e-9, "variant path drifted from control!"
print("OK: variant path == control path")

In [ ]:
variants = [("baseline", {})]
variants += [(f"size_gate_{th}", {"variant": "size_gate", "theta": th}) for th in SIZE_GATES]
variants += [(f"size_weighted_a{alpha}", {"variant": "size_weighted", "alpha": alpha}) for alpha in SIZE_ALPHAS]
variants += [(f"kth_{k}", {"variant": "kth", "k": k}) for k in KTH_LIST]
variants += [(f"conf_gate_{tau}", {"variant": "conf_gate", "tau": tau}) for tau in CONF_TAUS]
variants += [("conf_weighted", {"variant": "conf_weighted"})]

rows = []
for name, kw in variants:
    micro, macro, _, _ = eval_variants(pool_variant(views, kw["variant"], **{k: v for k, v in kw.items() if k != "variant"}), views)
    rows.append({"variant": name, "micro_auc": micro, "macro_auc": macro,
                 "delta_pts": (micro - micro_base) * 100})
    print(f"{name:26s} micro={micro*100:7.4f}%  macro={macro*100:7.4f}%  delta={(micro-micro_base)*100:+6.2f} pts")

part_b = pd.DataFrame(rows)
part_b.to_csv(OUTPUT_ROOT / "part_b_pooling_variants.csv", index=False)
print("saved:", OUTPUT_ROOT / "part_b_pooling_variants.csv")

### Reading Part B

- **baseline** must sit at 64.1421% (it did, or the cell above would have failed).
- A **size gate / size-weighted** gain ⇒ the argmin person on false positives is a small crowd
  person ⇒ size weighting is a real lever.
- A **conf gate / conf-weighted** gain ⇒ garbage (low-confidence) detections are the noise.
- **k-th smallest** is a probe: if k=2 jumps the AUC, a single noise person dominates the min —
  quantifying the mechanism — but it is *not* a candidate final design (it misses the lying-man
  case).
- Any variant that *loses* points is still informative: it shows the min over all people is
  already carrying real signal.

### Part B — argmin-person attribution

For the **top-1% false-positive frames** and the **true-positive frames**, report the size and
confidence of the argmin person (the person driving the frame score under baseline min-pooling).
This is the direct test of the background-crowd hypothesis: do false positives come from small
people?

In [ ]:
# Baseline pooled (smoothed) frame scores in control convention; anomaly = -score.
_, _, sc_arr_b, gt_arr_b = eval_variants(pool_variant(views, "baseline"), views)
anom_all = -np.concatenate(sc_arr_b)
gt_all = np.concatenate(gt_arr_b)

# Global FP threshold = top TOP_FP_FRAC fraction of *normal* frames by anomaly.
n_fp = max(MIN_FP_N, int((gt_all == 0).sum() * TOP_FP_FRAC))
fp_scores = np.sort(anom_all[gt_all == 0])[-n_fp:]
fp_thr = fp_scores.min()
print(f"FP threshold: top {n_fp} normal frames, anomaly > {fp_thr:.4f}")

rows = []
off = 0
for clip, v in views.items():
    T = v["gt"].shape[0]
    persons = v["persons"]
    P = np.stack([v["score"][p] for p in persons])
    S = np.stack([v["size"][p] for p in persons])
    C = np.stack([v["conf"][p] for p in persons])
    anom = -np.asarray(sc_arr_b[off:off + T])
    gt = v["gt"]
    for t in range(T):
        is_fp = (gt[t] == 0) and (anom[t] >= fp_thr)
        is_tp = (gt[t] == 1)
        if not (is_fp or is_tp):
            continue
        pid_row = int(np.argmin(P[:, t]))
        finite = np.isfinite(P[:, t])
        if not finite.any():
            rows.append({"clip": clip, "frame": t, "gt": int(gt[t]), "kind": "FP" if is_fp else "TP",
                         "argmin_person": -1, "argmin_size": np.nan, "argmin_conf": np.nan})
            continue
        rows.append({"clip": clip, "frame": t, "gt": int(gt[t]), "kind": "FP" if is_fp else "TP",
                     "argmin_person": int(persons[pid_row]),
                     "argmin_size": float(S[pid_row, t]),
                     "argmin_conf": float(C[pid_row, t])})
    off += T

attr = pd.DataFrame(rows)
attr.to_csv(OUTPUT_ROOT / "part_b_argmin_attribution.csv", index=False)
for kind in ["FP", "TP"]:
    sub = attr[attr["kind"] == kind]
    sz = sub["argmin_size"].dropna()
    cf = sub["argmin_conf"].dropna()
    print(f"{kind}: n={len(sub)}")
    if len(sz):
        print(f"  argmin size:  median={sz.median():6.1f}  p25={sz.quantile(0.25):6.1f}  p75={sz.quantile(0.75):6.1f}")
        print(f"  size < {CROWD_THETA}px: {(sz < CROWD_THETA).mean()*100:5.1f}%   size >= {SIZE_GATES[-1]}px: {(sz >= SIZE_GATES[-1]).mean()*100:5.1f}%")
    if len(cf):
        print(f"  argmin conf:  median={cf.median():.3f}   < {GARBAGE_CONF}: {(cf < GARBAGE_CONF).mean()*100:5.1f}%")
print("saved:", OUTPUT_ROOT / "part_b_argmin_attribution.csv")

### Reading the attribution

- If **FP argmin people are small** (median < 80 px) while TP argmin people are large, the
  background-crowd hypothesis is confirmed at the per-object level — the size-gate/weight
  numbers in the variant table are the expected gain.
- If FP argmin people are low-confidence, Part C (simulated stricter detection) is the lever.
- If FP argmin people are large *and* confident, the min is picking real-looking foreground
  people — the problem is upstream (identity continuity) or model capacity, and both Part B and
  Part C will be flat.

---
# Part C — Simulated stricter detection (JSON filtering, no re-extraction)

The alternative to scoring-side fixes is to prevent garbage at the source: raise the detection
bar, require keypoint quality, ignore tiny people, drop unstable tracks — *do not detect them*.
The tracked-person JSONs already carry per-frame detection confidence (`scores`) and per-keypoint
confidence, so stricter extraction is **simulated by deleting records** and re-running the
baseline. Every setting needs its own model pass (filtering changes the segments), but each pass
is ~20–40 s.

In [ ]:
def filter_clip_json(clip_dict, det_tau=None, kp_floor=None, size_floor=None, kp_conf=KP_CONF_FILTER):
    out = {}
    for pid, frames in clip_dict.items():
        kept = {}
        for fk, rec in frames.items():
            kps = np.array(rec["keypoints"]).reshape(-1, 3)
            conf_kp = kps[:, 2]
            if det_tau is not None and rec["scores"] < det_tau:
                continue
            if kp_floor is not None and (conf_kp >= kp_conf).sum() < kp_floor:
                continue
            if size_floor is not None:
                ys = kps[conf_kp >= KP_CONF_SIZE, 1]
                if len(ys) < 2 or (ys.max() - ys.min()) < size_floor:
                    continue
            kept[fk] = rec
        if kept:
            out[pid] = kept
    return out


def drop_short_tracks(clip_dict, min_len):
    return {pid: fr for pid, fr in clip_dict.items() if len(fr) >= min_len}


def write_filtered(pose_dir, out_dir, **filters):
    out_dir.mkdir(parents=True, exist_ok=True)
    for fn in sorted(p for p in os.listdir(pose_dir) if p.endswith("tracked_person.json")):
        with open(os.path.join(pose_dir, fn)) as f:
            clip = json.load(f)
        clip = filter_clip_json(clip, det_tau=filters.get("det_tau"),
                               kp_floor=filters.get("kp_floor"),
                               size_floor=filters.get("size_floor"))
        if filters.get("min_track"):
            clip = drop_short_tracks(clip, filters["min_track"])
        with open(out_dir / fn, "w") as f:
            json.dump(clip, f)
    return out_dir

print("filter helpers ready")

In [ ]:
# Build every filter setting, apply to test (scored) and train (coverage), re-run the baseline.
settings = []
for tau in PART_C_DET_TAUS:
    settings.append((f"det_conf_{tau}", dict(det_tau=tau)))
for k in PART_C_KP_FLOORS:
    settings.append((f"kp_floor_{k}", dict(kp_floor=k)))
for s in PART_C_SIZE_FLOORS:
    settings.append((f"size_floor_{s}", dict(size_floor=s)))
for l in PART_C_TRACK_LENS:
    settings.append((f"min_track_{l}", dict(min_track=l)))
settings.append(("combined", PART_C_COMBINED))

rows = []
for name, filters in settings:
    test_dir = write_filtered(POSE_TEST, OUTPUT_ROOT / f"part_c/{name}/test", **filters)
    train_dir = write_filtered(POSE_TRAIN, OUTPUT_ROOT / f"part_c/{name}/train", **filters)
    n_test = len(build_split_dataset(test_dir, 1))
    n_train = len(build_split_dataset(train_dir, 6))
    sc, ds, _ = run_control(pose_test_dir=test_dir)
    auc, _ = score_dataset(sc, ds["test"].metadata, args=build_args(pose_test_dir=test_dir))
    rows.append({"setting": name, "micro_auc": float(auc), "delta_pts": (float(auc) - micro_base) * 100,
                 "n_test_segs": int(n_test), "n_train_segs": int(n_train)})
    print(f"{name:16s} micro={auc*100:7.4f}%  delta={(auc-micro_base)*100:+6.2f} pts  test_segs={n_test:6d}  train_segs={n_train:6d}")

part_c = pd.DataFrame(rows)
part_c.to_csv(OUTPUT_ROOT / "part_c_filtering.csv", index=False)
print("saved:", OUTPUT_ROOT / "part_c_filtering.csv")

### Reading Part C

- **Coverage guard:** a filter that decimates train segments (e.g. `min_track_24` dropping most
  of training) is not viable even if its test AUC looks good — the model must still learn the
  normal distribution. Keep the smallest setting that keeps ≥ ~95% of train segments.
- A **positive delta** means the deleted person-frames were noise: the thresholds are already
  known, so the follow-up is a *real* re-extraction with them (not another guess).
- A **flat** Part C with a **positive** Part B size variant ⇒ the noise is *real small-person
  jitter* (confident detections), which only scoring-side size weighting removes.
- Both flat ⇒ identity continuity or model capacity; effort moves to the tracking path.

---
# Part D — Failure attribution and evidence

Turns the AUC numbers into a defensible story: *which objects* make which frames/clips score
badly. Every flagged frame decomposes as
`frame score → argmin person → track → JSON pose → size/confidence → reason code`.

Descriptive attribution (who drove the score — certain, from the argmin) is reported separately
from causal claims (what caused the AUC loss — established by the counterfactual probes below).

In [ ]:
def parse_clip_stats(json_path):
    """Per-person per-frame stats from the raw JSON: size, det conf, jitter, gaps, track length."""
    with open(json_path) as f:
        clip = json.load(f)
    stats = {}
    for pid, frames in clip.items():
        fks = sorted(frames.keys(), key=lambda x: int(x))
        ints = [int(fk) for fk in fks]
        per_frame = {}
        prev = None
        for fk in fks:
            rec = frames[fk]
            kps = np.array(rec["keypoints"]).reshape(-1, 3)
            conf = kps[:, 2]
            mask = conf >= KP_CONF_SIZE
            ys = kps[mask, 1]
            h = float(ys.max() - ys.min()) if mask.sum() >= 2 else float("nan")
            jit = float("nan")
            if prev is not None:
                both = mask & (prev[:, 2] >= KP_CONF_SIZE)
                if both.sum() > 0:
                    jit = float(np.linalg.norm(kps[both, :2] - prev[both, :2], axis=1).mean())
            per_frame[int(fk)] = {"h": h, "det": float(rec["scores"]), "jit": jit,
                                 "n_conf": int(mask.sum())}
            prev = kps
        gaps = [(a, b) for a, b in zip(ints, ints[1:]) if b - a > 1]
        stats[pid] = {"len": len(ints), "first": ints[0], "last": ints[-1],
                      "gaps": gaps, "per_frame": per_frame}
    return stats


def reason_code(st, pid, t, seg_len=SEG_LEN):
    """Machine-assigned reason for a flagged frame, given the argmin person's track stats."""
    if st is None or pid is None:
        return "NO_DET"
    info = st.get(str(pid)) or st.get(int(pid))
    if info is None:
        return "NO_DET"
    if info["len"] < seg_len:
        return "ID_SWITCH"          # newborn / reborn track
    for a, b in info["gaps"]:
        if a - seg_len <= t <= b + seg_len:
            return "OCCLUSION_GAP"  # track gap near this frame (behind a column/wall)
    pf = info["per_frame"].get(int(t))
    if pf is None:
        return "NO_DET"
    if pf["det"] < GARBAGE_CONF or pf["n_conf"] < GARBAGE_KP:
        return "GARBAGE_DET"
    if pf["h"] < CROWD_THETA and pf["jit"] > JITTER_HIGH:
        return "CROWD_JITTER"
    return "OTHER"

print("attribution helpers ready")

In [ ]:
# Per-clip report + reason-code histogram over FP frames and missed events (baseline pool).
clip_rows = []
fp_reasons = []
missed_rows = []

off = 0
for clip in sorted(views.keys()):
    v = views[clip]
    T = v["gt"].shape[0]
    gt = v["gt"]
    anom = -np.asarray(sc_arr_b[off:off + T])
    json_path = os.path.join(POSE_TEST, clip.replace(".npy", "") + "_vitpose_tracked_person.json")
    if not os.path.exists(json_path):
        hits = [p for p in os.listdir(POSE_TEST) if p.startswith(clip.replace(".npy", ""))]
        json_path = os.path.join(POSE_TEST, hits[0]) if hits else None
    stats = parse_clip_stats(json_path) if json_path else {}
    persons = v["persons"]
    P = np.stack([v["score"][p] for p in persons])
    S = np.stack([v["size"][p] for p in persons])

    # per-clip AUC + event detection check
    auc_c = roc_auc_score(gt, anom) if len(np.unique(gt)) > 1 else float("nan")
    events = []
    in_ev = False
    for t in range(T):
        if gt[t] == 1 and not in_ev:
            start = t; in_ev = True
        elif gt[t] == 0 and in_ev:
            events.append((start, t - 1)); in_ev = False
    if in_ev:
        events.append((start, T - 1))
    n_det = 0
    for (a, b) in events:
        peak = anom[a:b + 1].max()
        detected = peak >= fp_thr
        n_det += int(detected)
        if not detected:
            # classify the miss
            win = anom[a:b + 1]
            if not np.isfinite(P[:, a:b + 1]).any():
                code = "MISSED_NO_DET"
            else:
                pmax = int(np.argmax(np.nanmax(P[:, a:b + 1], axis=1)))
                sz = S[pmax, a:b + 1].max()
                code = "MISSED_NORMALIZED" if (np.isfinite(sz) and sz >= CROWD_THETA) else "MISSED_MASKED"
            missed_rows.append({"clip": clip, "event": (a, b), "reason": code})

    # FP frames in this clip
    for t in range(T):
        if gt[t] == 0 and anom[t] >= fp_thr:
            finite = np.isfinite(P[:, t])
            pid_row = int(np.argmin(P[:, t])) if finite.any() else None
            pid = persons[pid_row] if pid_row is not None else None
            fp_reasons.append({"clip": clip, "frame": t, "reason": reason_code(stats, pid, t)})

    clip_rows.append({"clip": clip, "auc": auc_c, "deficit_pts": (auc_control - auc_c) * 100,
                      "events": len(events), "events_detected": n_det})
    off += T

clip_rep = pd.DataFrame(clip_rows)
clip_rep.to_csv(OUTPUT_ROOT / "part_d_per_clip.csv", index=False)
fp_rep = pd.DataFrame(fp_reasons)
fp_rep.to_csv(OUTPUT_ROOT / "part_d_fp_reasons.csv", index=False)
missed_rep = pd.DataFrame(missed_rows)
missed_rep.to_csv(OUTPUT_ROOT / "part_d_missed_events.csv", index=False)

print(clip_rep.to_string(index=False))
print("\nFP reason histogram (top-1% false positives):")
print(fp_rep["reason"].value_counts().to_string())
print("\nMissed event reasons:")
print(missed_rep["reason"].value_counts().to_string() if len(missed_rep) else "(none)")
print("\nsaved 3 CSVs to", OUTPUT_ROOT)

### Part D — counterfactual probes (causation, not correlation)

1. **Delete-person (free):** remove the argmin person's scores at the flagged frame and re-pool.
   Other persons' segment scores are unchanged, so this is pure re-aggregation — if the FP
   disappears, that object *caused* it.
2. **Gap-interpolation (one extra pass):** fill occlusion gaps ≤ MISSED_GAP frames by linear
   keypoint interpolation, rebuild segments, re-score, and check whether MISSED events recover.

In [ ]:
# Probe 1: delete the argmin person at each top-K FP frame and re-pool (no model pass).
fp_df = fp_rep.sort_values("frame")
probe_rows = []
for _, r in fp_df.head(50).iterrows():
    clip, t = r["clip"], int(r["frame"])
    v = views[clip]
    persons = v["persons"]
    P = np.stack([v["score"][p] for p in persons])
    finite = np.isfinite(P[:, t])
    if not finite.any():
        continue
    pid_row = int(np.argmin(P[:, t]))
    orig = float(np.min(P[:, t]))
    P2 = P.copy(); P2[pid_row, t] = np.inf
    new = float(np.min(P2[:, t]))
    # anomaly rises when the noise person is removed (less anomalous => smaller -score => not FP)
    removed_fp = (-new) < fp_thr
    probe_rows.append({"clip": clip, "frame": t, "argmin": persons[pid_row],
                       "score_before": orig, "score_after": new, "fp_removed": removed_fp})

probe = pd.DataFrame(probe_rows)
probe.to_csv(OUTPUT_ROOT / "part_d_delete_person_probe.csv", index=False)
print(f"delete-person probe: {probe['fp_removed'].sum()}/{len(probe)} false positives removed by deleting the argmin person")

In [ ]:
# Probe 2: gap interpolation. Fill occlusion gaps <= MISSED_GAP in the test JSONs, re-score,
# and compare per-clip AUC on clips that had MISSED events.
affected = sorted(set(missed_rep["clip"])) if len(missed_rep) else []
print("clips with missed events:", affected)

def interpolate_gaps(clip_dict, max_gap=MISSED_GAP):
    out = {}
    for pid, frames in clip_dict.items():
        fks = sorted(frames.keys(), key=lambda x: int(x))
        ints = [int(fk) for fk in fks]
        new_frames = {}
        for a, b in zip(ints, ints[1:]):
            gap = b - a - 1
            if gap <= 0 or gap > max_gap:
                continue
            ka = np.array(frames[str(a)]["keypoints"]).reshape(-1, 3)
            kb = np.array(frames[str(b)]["keypoints"]).reshape(-1, 3)
            da = frames[str(a)]["scores"]; db = frames[str(b)]["scores"]
            for j in range(1, gap + 1):
                w = j / (gap + 1)
                interp = (1 - w) * ka + w * kb
                interp[:, 2] = np.maximum(ka[:, 2], kb[:, 2])  # keep confident kps confident
                new_frames[str(a + j)] = {"keypoints": interp.reshape(-1).tolist(),
                                          "scores": float((1 - w) * da + w * db)}
        if new_frames:
            merged = {**frames, **new_frames}
            out[pid] = dict(sorted(merged.items(), key=lambda x: int(x[0])))
        else:
            out[pid] = frames
    return out

if affected and RUN_GAP_PROBE is True:
    interp_dir = OUTPUT_ROOT / "part_c/gap_interp/test"
    interp_dir.mkdir(parents=True, exist_ok=True)
    for fn in sorted(p for p in os.listdir(POSE_TEST) if p.endswith("tracked_person.json")):
        with open(os.path.join(POSE_TEST, fn)) as f:
            clip = json.load(f)
        with open(interp_dir / fn, "w") as f:
            json.dump(interpolate_gaps(clip), f)
    sc_i, ds_i, _ = run_control(pose_test_dir=interp_dir)
    auc_i, _ = score_dataset(sc_i, ds_i["test"].metadata, args=build_args(pose_test_dir=interp_dir))
    print(f"gap-interp micro AUC: {auc_i*100:.4f}%   (baseline {auc_control*100:.4f}%)")
else:
    print("skipped: no missed events (or RUN_GAP_PROBE off)")

### Part D — two-stack comparison (optional)

Run the same FP reason histogram on the AlphaPose stack. If **both** stacks are dominated by the
same reason code (e.g. `CROWD_JITTER`), that is the cleanest possible explanation of the shared
64% plateau: the same mechanism under two different extractors.

In [ ]:
if ALPHAPOSE_POSE_ROOT is not None and ALPHAPOSE_POSE_ROOT.exists():
    ap_test = ALPHAPOSE_POSE_ROOT / "test"
    sc_a, ds_a, args_a = run_control(pose_test_dir=ap_test)
    auc_a, _ = score_dataset(sc_a, ds_a["test"].metadata, args=args_a)
    print(f"AlphaPose stack micro AUC: {auc_a*100:.4f}%")
    views_a = build_clip_views(sc_a, ds_a, args_a)
    _, _, sc_arr_a, gt_arr_a = eval_variants(pool_variant(views_a, "baseline"), views_a)
    anom_a = -np.concatenate(sc_arr_a)
    gt_a = np.concatenate(gt_arr_a)
    n_fp_a = max(MIN_FP_N, int((gt_a == 0).sum() * TOP_FP_FRAC))
    thr_a = np.sort(anom_a[gt_a == 0])[-n_fp_a:].min()
    ap_reasons = []
    off = 0
    for clip, v in views_a.items():
        T = v["gt"].shape[0]
        anom = -np.asarray(sc_arr_a[off:off + T])
        gt = v["gt"]
        json_path = os.path.join(ap_test, clip.replace(".npy", "") + "_alphapose_tracked_person.json")
        if not os.path.exists(json_path):
            hits = [p for p in os.listdir(ap_test) if p.startswith(clip.replace(".npy", ""))]
            json_path = os.path.join(ap_test, hits[0]) if hits else None
        stats = parse_clip_stats(json_path) if json_path else {}
        persons = v["persons"]
        P = np.stack([v["score"][p] for p in persons])
        for t in range(T):
            if gt[t] == 0 and anom[t] >= thr_a:
                finite = np.isfinite(P[:, t])
                pid_row = int(np.argmin(P[:, t])) if finite.any() else None
                pid = persons[pid_row] if pid_row is not None else None
                ap_reasons.append(reason_code(stats, pid, t))
        off += T
    ap_hist = pd.Series(ap_reasons).value_counts()
    ap_hist.to_csv(OUTPUT_ROOT / "part_d_fp_reasons_alphapose.csv")
    print("AlphaPose FP reasons:\n", ap_hist.to_string())
else:
    print("AlphaPose root not found; skipped. Set ALPHAPOSE_POSE_ROOT or ignore.")

---
# Summary — decision rule

All artifacts are saved under `OUTPUT_ROOT`. This cell applies the decision rule and writes the
run manifest (checkpoint, args, every threshold) so the whole diagnostic is reproducible.

In [ ]:
best_b = part_b[part_b["variant"] != "baseline"].sort_values("delta_pts", ascending=False).iloc[0]
best_c = part_c.sort_values("delta_pts", ascending=False).iloc[0] if len(part_c) else None

print("=" * 70)
print(f"Control micro AUC           : {auc_control*100:.4f}%")
print(f"Best Part B pooling variant : {best_b['variant']}  delta={best_b['delta_pts']:+.2f} pts")
if best_c is not None:
    print(f"Best Part C filter          : {best_c['setting']}  delta={best_c['delta_pts']:+.2f} pts")
print("=" * 70)
print("\nReason histogram (ViTPose FP frames):")
print(fp_rep["reason"].value_counts().to_string())

decision = []
if best_c is not None and best_c["delta_pts"] >= 1.0:
    decision.append("PART C CONFIRMED: simulated stricter detection gains >= 1 pt. Re-extract with "
                    f"the {best_c['setting']} thresholds (already known from the simulation).")
if best_b["delta_pts"] >= 1.0:
    decision.append("PART B CONFIRMED: pooling variant gains >= 1 pt. Design the full scoring "
                    "experiment (confidence gating + size-weighted pooling + seg_len/smoothing).")
if not decision:
    decision.append("FLAT: scoring aggregation and detection filtering are ruled out cheaply. "
                    "Constraint is identity continuity (BoT-SORT/ReID + track stitching) or model "
                    "capacity (STG-NF tuning).")
print("\nDECISION:")
for d in decision:
    print("  -", d)

# ---- run manifest ----
manifest = {
    "checkpoint": str(CHECKPOINT),
    "control_micro_auc": float(auc_control),
    "thresholds": {
        "seg_len": SEG_LEN, "smooth_sigma": SMOOTH_SIGMA, "kp_conf_size": KP_CONF_SIZE,
        "size_gates": SIZE_GATES, "size_ref": SIZE_REF, "size_alphas": SIZE_ALPHAS,
        "kth_list": KTH_LIST, "conf_taus": CONF_TAUS,
        "part_c_det_taus": PART_C_DET_TAUS, "part_c_kp_floors": PART_C_KP_FLOORS,
        "part_c_size_floors": PART_C_SIZE_FLOORS, "part_c_track_lens": PART_C_TRACK_LENS,
        "part_c_combined": PART_C_COMBINED, "kp_conf_filter": KP_CONF_FILTER,
        "top_fp_frac": TOP_FP_FRAC, "min_fp_n": MIN_FP_N, "crowd_theta": CROWD_THETA,
        "jitter_high": JITTER_HIGH, "garbage_conf": GARBAGE_CONF, "garbage_kp": GARBAGE_KP,
        "missed_gap": MISSED_GAP,
    },
    "part_b_best": best_b.to_dict(),
    "part_c_best": best_c.to_dict() if best_c is not None else None,
    "fp_reason_histogram": fp_rep["reason"].value_counts().to_dict(),
    "missed_events": missed_rep["reason"].value_counts().to_dict() if len(missed_rep) else {},
}
with open(OUTPUT_ROOT / "run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print("\nmanifest saved:", OUTPUT_ROOT / "run_manifest.json")
print("\nAll artifacts:")
for p in sorted(OUTPUT_ROOT.rglob("*.csv")) + sorted(OUTPUT_ROOT.rglob("*.json")):
    print(" ", p.relative_to(OUTPUT_ROOT))